# Hypothesis 1: Orthogonality

Tests whether the semantic direction vectors for different demographic attributes (gender, race, and their intersections) are approximately orthogonal in CLIP text-embedding space. Each direction vector is computed as the mean difference between the embedding of `"<profession>"` and `"<attribute> <profession>"` prompts, averaged over a fixed set of professions. Near-zero off-diagonal cosine similarity in the heatmaps below supports the orthogonality hypothesis that underlies the paper's additive embedding-arithmetic framework.

In [ ]:
import re
import os
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
from tqdm import tqdm

from VLMAdapter import PrismVLMTextEncoder

prism = PrismVLMTextEncoder("openai/clip-vit-large-patch14", adapter_type="identity")

professions = ["Nurse", "Doctor", "Engineer", "Teacher", "Scientist", "Chef", "Police Officer", "CEO", "Artist", "Construction Worker"]
race_attrs = ["white", "black", "asian", "indian"]
gender_attrs = ["male", "female"]
intersection = [f"{race} {gender}" for race in race_attrs for gender in gender_attrs]


# Article helper
def choose_article(phrase):
    first_word = re.findall(r'\b\w+\b', phrase)[0]
    return 'An' if first_word[0].lower() in 'aeiou' else 'A'


def create_semantic_vector_lookup(professions, races, genders, encoder):
    prompts = {}
    for prof in professions:
        prompts[prof] = f"a photo portrait of {choose_article(prof)} {prof}"

        for race in races:
            prompts[f'{race} {prof}'] = f"a photo portrait of {choose_article(race)} {race} {prof}"
        for gender in genders:
            prompts[f'{gender} {prof}'] = f"a photo portrait of {choose_article(gender)} {gender} {prof}"
        for race in races:
            for gender in genders:
                prompts[f'{race} {gender} {prof}'] = f"a photo portrait of {choose_article(race)} {race} {gender} {prof}"

    print("\nCalculating embeddings for all prompts...")
    embeddings = {}
    for key, prompt in tqdm(prompts.items(), desc="Encoding Prompts"):
        embeddings[key] = encoder([prompt])

    semantic_vectors = {}
    intersection = [f"{race} {gender}" for race in races for gender in genders]
    all_attributes = races + genders + intersection

    for attr in tqdm(all_attributes):
        semantic_vectors[attr] = {}
        for prof in professions:
            base_embedding = embeddings[prof]
            attr_embedding = embeddings[f"{attr} {prof}"]
            diff_vector = attr_embedding - base_embedding
            semantic_vectors[attr][prof] = diff_vector

    print("\n--- Semantic Vector Lookup Table Created Successfully ---")
    return semantic_vectors


def average_semantic_attribute_vectors(semantic_vectors):
    averages = {}
    for k in semantic_vectors.keys():
        embs = torch.cat([semantic_vectors[k][p] for p in semantic_vectors[k].keys()]).mean(dim=0)
        averages[k] = embs
    return averages


semantic_vectors = create_semantic_vector_lookup(professions, race_attrs, gender_attrs, prism)
semantic_averages = average_semantic_attribute_vectors(semantic_vectors)

In [ ]:
def get_heatmap_interattrib(targets, title, ax, cbar):
    tensor_stack = torch.stack([F.normalize(semantic_averages[k], p=2, dim=-1) for k in targets]).squeeze().to(torch.float32)
    similarity_matrix = (tensor_stack @ tensor_stack.T).clip(max=1.0)
    similarity_matrix = similarity_matrix.cpu().numpy()

    sns.heatmap(
        similarity_matrix,
        xticklabels=targets,
        yticklabels=targets,
        annot=True,
        fmt=".2f",
        vmin=0, vmax=1,
        cmap="RdYlGn",
        cbar=False,
        ax=ax
    )


fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))
get_heatmap_interattrib(gender_attrs, "genders", ax[1], False)
get_heatmap_interattrib(race_attrs, "Races", ax[2], False)
get_heatmap_interattrib(intersection, "intersection", ax[0], True)
x_ticks = [f"{w.split(' ')[0]}\n{w.split(' ')[1]}" for w in intersection]
ax[0].set_xticklabels(x_ticks, rotation=90)
ax[0].set_yticklabels(x_ticks, rotation=0)
ax[1].set_title("Gender")
ax[2].set_title("Race")
ax[0].set_title("Race x Gender intersection")
fig.tight_layout()
fig.savefig("hypothesis_orthogonality.pdf")
plt.show()